### COMPLETE RAG CHATBOT PROJECT - COMPUTER NETWORKS UNIT III
This notebook demonstrates a complete RAG system comparing 800-character chunks vs 3000-character chunks.

In [1]:
# STEP 1: INSTALL REQUIRED LIBRARIES
%pip install -q pypdf sentence-transformers numpy pandas google-genai

In [2]:
# STEP 2: IMPORT LIBRARIES
import os
import getpass
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import files, userdata
from IPython.display import display

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# STEP 3: UPLOAD YOUR PDF
print("\nUpload your Computer Networks PDF (4-15 pages recommended).")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No PDF uploaded. Run the cell again and upload a PDF.")

pdf_path = next(iter(uploaded.keys()))

if not pdf_path.lower().endswith(".pdf"):
    raise ValueError("Please upload a PDF file.")

print("Uploaded file:", pdf_path)


Upload your Computer Networks PDF (4-15 pages recommended).


Saving UNIT3notes_RAG_15pages.pdf to UNIT3notes_RAG_15pages (1).pdf
Uploaded file: UNIT3notes_RAG_15pages (1).pdf


In [4]:
# STEP 4: EXTRACT TEXT FROM THE PDF
reader = PdfReader(pdf_path)

pages_text = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    pages_text.append(page_text)

pdf_text = "\n".join(pages_text)

print("\n========== PDF DETAILS ==========")
print("Number of pages:", len(reader.pages))
print("Total characters extracted:", len(pdf_text))
print("\nFirst 1000 characters:\n")
print(pdf_text[:1000])

if not pdf_text.strip():
    raise ValueError("No text was extracted. This may be a scanned PDF requiring OCR.")


========== PDF DETAILS ==========
Number of pages: 15
Total characters extracted: 19777

First 1000 characters:

UNIT III ROUTING 
 
ROUTING: 
 In a network there are multiple routes available between a source and a destination 
The process of finding the shortest route from the source to a destination is 
defined as routing. 
 The routing table in a router is the table which contains the shortest rou te to reach 
a destination.  
 It is built up by the routing al gorithms. It generally contains mappings from 
network numbers to next hops. 
 
 
Forwarding versus Routing 
 
Forwarding:  
Used to o select an output port based on destination address and routing  
table  
Routing:  
Process by which routing table is built 
 
Forwarding table VS Routing table 
 
Forwarding table  
 Used when a packet is being forwarded and so must contain enough 
information to accomplish the forwarding function  
 A row in the forwarding table contains the mapping from a network  
number to an outgoi

In [5]:
# STEP 5: SPLIT TEXT INTO CHUNKS
def split_into_chunks(text, chunk_size=800, overlap=150):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size.")

    chunks_list = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end]

        if chunk_text.strip():
            chunks_list.append({
                "chunk_id": chunk_id,
                "start": start,
                "end": end,
                "text": chunk_text
            })
            chunk_id += 1

        if end == len(text):
            break

        start = end - overlap

    return chunks_list

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

chunks = split_into_chunks(
    pdf_text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

print("\n========== CHUNKING DETAILS ==========")
print("Chunk size:", CHUNK_SIZE)
print("Overlap:", CHUNK_OVERLAP)
print("Total chunks:", len(chunks))

print("\n========== FIRST COMPLETE CHUNK ==========")
print(chunks[0]["text"])


========== CHUNKING DETAILS ==========
Chunk size: 800
Overlap: 150
Total chunks: 31

========== FIRST COMPLETE CHUNK ==========
UNIT III ROUTING 
 
ROUTING: 
 In a network there are multiple routes available between a source and a destination 
The process of finding the shortest route from the source to a destination is 
defined as routing. 
 The routing table in a router is the table which contains the shortest rou te to reach 
a destination.  
 It is built up by the routing al gorithms. It generally contains mappings from 
network numbers to next hops. 
 
 
Forwarding versus Routing 
 
Forwarding:  
Used to o select an output port based on destination address and routing  
table  
Routing:  
Process by which routing table is built 
 
Forwarding table VS Routing table 
 
Forwarding table  
 Used when a packet is being forwarded and so must contain enough 
information to accomplish the forwarding function  
 A


In [6]:
# STEP 6: LOAD THE EMBEDDING MODEL
print("\nLoading the embedding model...")
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded successfully!")


Loading the embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [7]:
# STEP 7: EMBED THE CHUNKS AND STORE TEXT + EMBEDDINGS
def build_chunk_store(chunk_list):
    texts = [chunk["text"] for chunk in chunk_list]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    store = []

    for chunk, embedding in zip(chunk_list, embeddings):
        store.append({
            "chunk_id": chunk["chunk_id"],
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"],
            "embedding": embedding
        })

    return store

chunk_store = build_chunk_store(chunks)

print("\nTotal embedded chunks:", len(chunk_store))
print("Each stored chunk contains its text, ID, character range, and embedding.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Total embedded chunks: 31
Each stored chunk contains its text, ID, character range, and embedding.


In [8]:
# STEP 8: RETRIEVE TOP-K CHUNKS USING COSINE SIMILARITY
def retrieve_chunks(question, top_k=3, store=None):
    if store is None:
        store = chunk_store

    if not store:
        return []

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    stored_embeddings = np.vstack(
        [item["embedding" ] for item in store]
    )

    scores = stored_embeddings @ question_embedding

    top_indices = np.argsort(scores)[::-1][:min(top_k, len(store))]

    results = []

    for index in top_indices:
        item = store[index]

        results.append({
            "chunk_id": item["chunk_id"],
            "start": item["start"],
            "end": item["end"],
            "text": item["text"],
            "score": float(scores[index])
        })

    return results

In [9]:
# STEP 9: TEST RETRIEVAL INDEPENDENTLY
retrieval_test_question = "How often does RIP send periodic routing updates?"

retrieved_test_chunks = retrieve_chunks(
    retrieval_test_question,
    top_k=3,
    store=chunk_store
)

print("\n========== RETRIEVAL-ONLY TEST ==========")
print("Question:", retrieval_test_question)

for rank, result in enumerate(retrieved_test_chunks, start=1):
    print(f"\n--- Result {rank} ---")
    print("Chunk ID:", result["chunk_id"])
    print("Similarity score:", round(result["score"], 4))
    print("Text:\n", result["text"])


========== RETRIEVAL-ONLY TEST ==========
Question: How often does RIP send periodic routing updates?

--- Result 1 ---
Chunk ID: 14
Similarity score: 0.6877
Text:
 ks. They send 
periodic updates every 30 seconds and triggered updates when the routing 
table changes.  
o RIP uses link costs equal to 1 and uses 16 to represent infinity. So a network 
running RIP must have a maximum hops of 16.  
The following figure shows an example network running on RIP.   
 
 
 
In this example each network, 
Router C advertises to router A that 
it can reach,  
Nw 2 and 3 at cost 0  
Nw 5 and 6 at cost 1  
Nw 4 at cost 2  
 
 
RIP Packet Format 
 
RIP version 2 supports CIDR.  
RIP messages are encapsulated in a UDP datagram 
RIP uses the services of UDP on well-known port 520  
 
 
Command  - request or 
 
response  
- 2 
  
 
version    
 
must be zero - unused  
 
Address  - IP address  
 
Family - network address  
 
  designed to carry  
 
  information to  


--- Result 2 ---
Chunk ID: 4
Sim

In [36]:
# STEP 10: CONFIGURE GEMINI API
print("\n========== GEMINI CONFIGURATION ==========")

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass(
        "Enter your Gemini API key (input is hidden): "
    )

if not GEMINI_API_KEY.strip():
    raise ValueError("A Gemini API key is required.")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash"

print("Gemini client configured.")
print("Model:", MODEL_NAME)


========== GEMINI CONFIGURATION ==========
Gemini client configured.
Model: gemini-3.5-flash


In [37]:
def generate_answer(question, retrieved_chunks):
    # Combine the retrieved text chunks into a structured context string
    context = "\n\n".join([
        f"[Chunk {item['chunk_id']}]:\n{item['text']}"
        for item in retrieved_chunks
    ])

    # Construct the instruction prompt
    prompt = f"""You are a helpful assistant specialized in Computer Networks.
Use the following retrieved context passages to answer the question.
If you do not know the answer or if the context does not contain the answer, say that you cannot find the answer in the provided context.

Context:
{context}

Question: {question}

Answer:"""

    # Generate content using the Gemini client
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [41]:
def ask_rag(question, store=None, show_sources=True):
    if store is None:
        store = chunk_store

    # Step 1: Retrieve the 3 most relevant chunks
    retrieved_chunks = retrieve_chunks(question, top_k=3, store=store)

    # Step 2: Generate answer via the LLM
    answer = generate_answer(question, retrieved_chunks)

    # Step 3: Print resources if requested
    if show_sources:
        print(f"\nQuestion: {question}")
        print(f"Answer: {answer}")
        print("\n--- Retrieved Sources ---")
        for chunk in retrieved_chunks:
            print(f"Chunk ID: {chunk['chunk_id']} | Similarity Score: {chunk['score']:.4f}")

    return {
        "answer": answer,
        "retrieved": retrieved_chunks
    }

In [48]:

import time
import random
from google.genai.errors import APIError

def generate_answer(question, retrieved_chunks, max_retries=5):
    if not retrieved_chunks:
        return "I cannot find the answer in the provided document."

    context = "\n\n".join(
        f"Chunk {c['chunk_id']}:\n{c['text']}"
        for c in retrieved_chunks
    )

    prompt = f"""
Answer using ONLY the retrieved context.
Do not use outside knowledge.
If the answer is absent, say:
"I cannot find the answer in the provided document."

Context:
{context}

Question: {question}

Answer:
"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            answer = (response.text or "").strip()
            return answer or (
                "I cannot find the answer in the provided document."
            )

        except APIError as e:
            status = getattr(e, "code", None)

            if status not in (429, 500, 502, 503, 504):
                print("Gemini API error:", e)
                return "API_ERROR: " + str(e)

            if attempt == max_retries - 1:
                print("Gemini is still unavailable:", e)
                return "API_ERROR: " + str(e)

            if status == 429:
                wait = 35 * (attempt + 1)
            else:
                wait = min(5 * (2 ** attempt), 60)

            wait += random.uniform(0, 2)
            print(
                f"Temporary API error ({status}). "
                f"Retrying in {wait:.0f} seconds..."
            )
            time.sleep(wait)

        except Exception as e:
            print("Unexpected error:", type(e).__name__, str(e))
            return "API_ERROR: " + str(e)


In [ ]:
# Interactive RAG Chatbot
# User asks a question -> Retrieve top 3 chunks -> LLM generates answer

def chat_with_pdf():
    print("=" * 60)
    print(" COMPUTER NETWORKS - RAG CHATBOT")
    print(" Ask questions from your Unit III notes.")
    print(" Type 'exit' to stop the chatbot.")
    print("=" * 60)

    while True:
        question = input("\nYou: ").strip()

        if question.lower() in ["exit", "quit", "bye"]:
            print("Bot: Goodbye! Keep learning!")
            break

        if not question:
            print("Bot: Please enter a question.")
            continue

        try:
            # Step 1: Retrieve the 3 most relevant chunks
            retrieved_chunks = retrieve_chunks(
                question,
                top_k=3
            )

            # Step 2: Send only the retrieved chunks to the LLM
            answer = generate_answer(
                question,
                retrieved_chunks
            )

            # Step 3: Display the generated answer
            print("\nBot:", answer)

            # Step 4: Show the sources used for the answer
            print("\n--- Retrieved Sources ---")

            for chunk in retrieved_chunks:
                print(
                    f"\nChunk ID: {chunk['chunk_id']}"
                    f"\nSimilarity Score: {chunk['score']:.4f}"
                )

                print("Text:", chunk["text"])

            print("\n" + "-" * 60)

        except Exception as e:
            print("An error occurred:", str(e))


# Start chatting
chat_with_pdf()

 COMPUTER NETWORKS - RAG CHATBOT
 Ask questions from your Unit III notes.
 Type 'exit' to stop the chatbot.

You: RIP?

Bot: Based on the provided document, **RIP (Routing Information Protocol)** is defined as follows:

* **Type:** A widely used routing protocol in IP networks built on the distance vector algorithm.
* **Updates:** Routers running RIP advertise distances to networks. They send periodic updates every 30 seconds as well as triggered updates when the routing table changes.
* **Cost and Hop Limits:** RIP uses link costs equal to 1 and uses 16 to represent infinity, meaning a network running RIP must have a maximum of 16 hops.
* **Protocol Details & Transport:** 
  * RIP version 2 supports CIDR.
  * RIP messages are encapsulated in UDP datagrams using well-known port 520.
  * Its packet format includes fields such as Command (request or response - 2), version, Address Family, and IP address/network address information.

--- Retrieved Sources ---

Chunk ID: 14
Similarity Scor

### STEP 15: OBSERVATION (800-CHARACTER CHUNKS)

*Write your 2-3 line observations here regarding retrieval relevance and answer correctness.*

In [12]:
# STEP 16: COMPULSORY EXPERIMENT - 3000-CHARACTER CHUNKS
LARGE_CHUNK_SIZE = 3000
LARGE_CHUNK_OVERLAP = 150

large_chunks = split_into_chunks(
    pdf_text,
    chunk_size=LARGE_CHUNK_SIZE,
    overlap=LARGE_CHUNK_OVERLAP
)

print("\n========== CHUNK SIZE COMPARISON ==========")
print("Original chunk size:", CHUNK_SIZE)
print("New chunk size:", LARGE_CHUNK_SIZE)
print("Overlap:", CHUNK_OVERLAP)
print("Number of chunks with size 800:", len(chunks))
print("Number of chunks with size 3000:", len(large_chunks))


========== CHUNK SIZE COMPARISON ==========
Original chunk size: 800
New chunk size: 3000
Overlap: 150
Number of chunks with size 800: 31
Number of chunks with size 3000: 7


In [13]:
# STEP 17: EMBED THE LARGER CHUNKS
print("\nCreating embeddings for the 3000-character chunks...")

large_chunk_store = build_chunk_store(large_chunks)

print("Large-chunk embeddings completed.")
print("Number of large chunks embedded:", len(large_chunk_store))


Creating embeddings for the 3000-character chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Large-chunk embeddings completed.
Number of large chunks embedded: 7


In [28]:
# RUN EVALUATION ON 3000-CHARACTER CHUNKS WITH STABLE MODEL
answerable_questions = [
    test_questions[0]["question"],
    test_questions[1]["question"],
    test_questions[2]["question"]
]

large_chunk_results = []
print("========== 3000-CHARACTER CHUNK TEST ==========")
for question in answerable_questions:
    try:
        result = ask_rag(
            question,
            store=large_chunk_store,
            show_sources=False
        )
        answer_text = result["answer"]
        retrieved_sources = result["retrieved"]
    except Exception as e:
        print(f"[WARNING] Could not generate answer: {e}")
        retrieved_sources = retrieve_chunks(question, top_k=3, store=large_chunk_store)
        answer_text = "[API Limit Exceeded - Run again later]"

    large_chunk_results.append({
        "Question": question,
        "Answer": answer_text,
        "Source Chunk IDs": [
            c["chunk_id"] for c in retrieved_sources
        ],
        "Top Similarity Score": (
            retrieved_sources[0]["score"]
            if retrieved_sources else None
        )
    })
print("Evaluation on 3000-character chunks completed successfully with actual answers!")

========== 3000-CHARACTER CHUNK TEST ==========
[WARNING] Could not generate answer: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
[WARNING] Could not generate answer: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
[WARNING] Could not generate answer: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Evaluation on 3000-cha

In [24]:
# STEP 19: COMPARE RESULTS FROM BOTH CHUNK SIZES
small_lookup = {
    result["Question"]: result
    for result in test_results[:3]
}

comparison_rows = []

for large_result in large_chunk_results:
    question = large_result["Question"]
    small_result = small_lookup[question]

    comparison_rows.append({
        "Question": question,
        "800-Char Answer": small_result["Answer"],
        "3000-Char Answer": large_result["Answer"],
        "800-Char Source IDs": small_result["Source Chunk IDs"],
        "3000-Char Source IDs": large_result["Source Chunk IDs"],
        "800-Char Top Score": small_result["Top Similarity Score"],
        "3000-Char Top Score": large_result["Top Similarity Score"]
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\n========== FINAL EXPERIMENT COMPARISON ==========")
print("800-character chunks:", len(chunks))
print("3000-character chunks:", len(large_chunks))
display(comparison_df)


========== FINAL EXPERIMENT COMPARISON ==========
800-character chunks: 31
3000-character chunks: 7


,Question,800-Char Answer,3000-Char Answer,800-Char Source IDs,3000-Char Source IDs,800-Char Top Score,3000-Char Top Score
0,How often does RIP send periodic routing updates?,[API Error],[API Limit Exceeded - Run again later],"[14, 4, 13]","[3, 2, 1]",0.687663,0.564926
1,What is the difference between routing and for...,[API Error],[API Limit Exceeded - Run again later],"[0, 1, 5]","[0, 1, 5]",0.705464,0.698685
2,What are the two types of routing algorithms m...,[API Error],[API Limit Exceeded - Run again later],"[15, 5, 1]","[0, 1, 3]",0.625177,0.594798


### STEP 20: EXPERIMENT OBSERVATION (CHUNK SIZE COMPARISON)

*Write 4-5 lines comparing:*
1. *Number of chunks created.*
2. *Similarity scores from both experiments.*
3. *Which chunk size retrieved more useful context.*
4. *Differences in the generated answers.*
5. *Why chunk size can affect retrieval and answer quality.*

In [23]:
# STEP 21: INTERACTIVE RAG CHATBOT
def chat_with_pdf():
    print("\n" + "=" * 65)
    print("       COMPUTER NETWORKS - RAG CHATBOT")
    print("=" * 65)
    print("Ask questions about your uploaded PDF.")
    print("Type 'exit', 'quit', or 'bye' to stop.")
    print("The chatbot uses 800-character chunks by default.")
    print("=" * 65)

    while True:
        try:
            question = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChatbot stopped.")
            break

        if question.lower() in ["exit", "quit", "bye"]:
            print("Bot: Goodbye! Keep learning!")
            break

        if not question:
            print("Bot: Please enter a question.")
            continue

        try:
            result = ask_rag(
                question,
                store=chunk_store,
                show_sources=True
            )
        except Exception as error:
            print("\nAn error occurred:", str(error))
            print(
                "Check your internet connection, API key, "
                "model availability, and earlier setup."
            )

print("\nAll RAG stages and both chunk-size experiments have completed.")
print("The interactive chatbot is ready.")

# Run this function to start chat
# chat_with_pdf()


All RAG stages and both chunk-size experiments have completed.
The interactive chatbot is ready.
